In [14]:
import pandas as pd, numpy as np, textstat, json, textdistance, textwrap, os, re
import plotly_express as px
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from evaluate import load
from azure.ai.evaluation import F1ScoreEvaluator, BleuScoreEvaluator, GleuScoreEvaluator, RougeScoreEvaluator, RougeType, MeteorScoreEvaluator
from transformers import pipeline
from IPython.display import clear_output
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.figure_factory as ff

import plotly.graph_objects as go
import plotly.express as px
import matplotlib.colors as mcolors

In [15]:
with open("llm_explanations/xaiqb_human_verbose.json") as f:
    xaiqb_human = json.load(f)

llm_responses = {}
for folder in ["run_1", "run_2", "run_3"]:
    llm_responses.update({folder: {}})
    for file in [x for x in os.listdir(f"llm_explanations/{folder}") if re.search(".json", x) is not None]:
        with open(f"llm_explanations/{folder}/{file}") as f:
                llm_responses[folder].update({file.replace(".json", ""): json.load(f)})


In [16]:
# Load all evaluation models
rouge_score = RougeScoreEvaluator(rouge_type = RougeType.ROUGE_L) 
bertscore = load("bertscore")
similarity_model = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

# f1_score = F1ScoreEvaluator()
# bleu_score = BleuScoreEvaluator()
# gleu_score = GleuScoreEvaluator()
# meteor_score = MeteorScoreEvaluator()

# nli = pipeline("text-classification", model = "roberta-large-mnli", top_k = None)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1274.72it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1145.06it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
# def nli_label(premise, hypothesis):
#     # Premise entails Hypothesis? (using MNLI framing)
#     # We return the best label among entailment/neutral/contradiction
#     out = nli({"text": premise, "text_pair": hypothesis})[0]
#     # best = max(out, key=lambda x: x["score"]); return best["label"], float(best["score"])
#     return out["label"], float(out["score"])

# Example scoring for one Q/A pair
# def score_pair(pred, ref):
#     entail_ph, conf_ph = nli_label(pred, ref)  # does pred entail ref?
#     entail_hp, conf_hp = nli_label(ref, pred)  # does ref entail pred?
#     return (entail_ph, round(conf_ph, 3)), (entail_hp, round(conf_hp, 3))

In [18]:
def similarity_scores(
        response, 
        ground_truth,
        include_terms,
        category = None, 
        question = None, 
        model_name = None
        ):

    bert_results = bertscore.compute(predictions = [response], references = [ground_truth], lang = "en")
    sbert_results = util.cos_sim(similarity_model.encode(response, normalize_embeddings = True, convert_to_tensor = True),
                                 similarity_model.encode(ground_truth, normalize_embeddings = True, convert_to_tensor = True)
                                 ).tolist()[0][0]
        

    # nli_results = score_pair(response, ground_truth)

    similarity_scores = pd.DataFrame({
        "Model": model_name,
        "Category": category,
        "Question": question,
        "Rouge-L IU Recall": [np.mean([rouge_score(response = response, ground_truth = x)["rouge_recall"] for x in eval(include_terms)])],
        "BERT IU Recall": [np.mean([bertscore.compute(predictions = [response], references = [x], lang = "en")["recall"][0] for x in eval(include_terms)])],
        "BERT Precision": bert_results["precision"][0],
        "BERT Recall": bert_results["recall"][0],
        "BERT F1": bert_results["f1"][0],
        "SBERT": sbert_results,

        # "F1": [f1_score(response = response, ground_truth = ground_truth)["f1_score"]],
        # "Bleu": [bleu_score(response = response, ground_truth = ground_truth)["bleu_score"]],
        # "Gleu": [gleu_score(response = response, ground_truth = ground_truth)["gleu_score"]],
        # "METEOR": [meteor_score(response = response, ground_truth = ground_truth)["meteor_score"]],
        # "Rouge-L F1": [rougel_score(response = response, ground_truth = ground_truth)["rouge_f1_score"]],
        # "Rouge-L Precision": [rougel_score(response = response, ground_truth = ground_truth)["rouge_precision"]],
        # "Pred Entails Ref": [nli_results[0]],
        # "Ref Entails Pred":[nli_results[1]],
        
        # "Ground Truth": [ground_truth],
        # "Response": [response]
    })
    return similarity_scores.round(3)

In [19]:
def pull_similarity_scores(
        model_name, 
        llm_response, 
        human_response = xaiqb_human,
        human_response_type = ["answer", "include_terms"][0]
        ):
    
    similarity_scores_df = []
    for category in human_response["questions"].keys():
        for question in human_response["questions"][category].keys():
            similarity_scores_df += [
                similarity_scores(
                    response = llm_response["questions"][category][question]["answer"],
                    ground_truth = human_response["questions"][category][question][human_response_type],
                    include_terms = human_response["questions"][category][question]["include_terms"],
                    category = category,
                    question = question,
                    model_name = model_name
            )]
    
    return pd.concat(similarity_scores_df)

In [20]:
def prettyName(x):
    return x.replace("xaiqb_", "").replace("_", " ").title().replace("gpt", "GPT")

In [21]:
similarity_df = pd.concat([
    pd.concat([pull_similarity_scores(x, y, human_response_type = "answer") for x, y in llm_responses["run_1"].items()]),
    pd.concat([pull_similarity_scores(x, y, human_response_type = "answer") for x, y in llm_responses["run_2"].items()]),
    pd.concat([pull_similarity_scores(x, y, human_response_type = "answer") for x, y in llm_responses["run_3"].items()])
    ], keys = ["Run 1", "Run 2", "Run 3"]
    ).reset_index(level = 0).rename(columns = {"level_0": "Run"}).assign(Model = lambda x: x["Model"].apply(lambda x: prettyName(x)))

similarity_df

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 1654.14it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,Run,Model,Category,Question,Rouge-L IU Recall,BERT IU Recall,BERT Precision,BERT Recall,BERT F1,SBERT
0,Run 1,Claude Opus 4.6,Data,What kind of data was the system trained on?,1.000,0.859,0.859,0.878,0.868,0.730
0,Run 1,Claude Opus 4.6,Data,What is the source of the training data?,0.952,0.911,0.888,0.944,0.915,0.934
0,Run 1,Claude Opus 4.6,Data,How were the labels/ground-truth produced?,1.000,0.864,0.874,0.909,0.891,0.803
0,Run 1,Claude Opus 4.6,Data,What is the sample size of the training data?,0.500,0.794,0.859,0.903,0.880,0.406
0,Run 1,Claude Opus 4.6,Data,What dataset(s) is the system NOT using?,0.400,0.858,0.815,0.873,0.843,0.713
...,...,...,...,...,...,...,...,...,...,...
0,Run 3,ChatGPT 5.2 Extended Thinking,Others,Why is the system using or not using a given a...,0.357,0.857,0.869,0.864,0.866,0.764
0,Run 3,ChatGPT 5.2 Extended Thinking,Others,What are the results of other people using the...,1.000,0.861,0.837,0.859,0.848,0.396
0,Run 3,ChatGPT 5.2 Extended Thinking,Others,Who is responsible for this system? Who are th...,1.000,0.925,0.947,0.961,0.954,0.791
0,Run 3,ChatGPT 5.2 Extended Thinking,UI,What are the affordances of the system?,0.300,0.851,0.841,0.904,0.871,0.652


In [22]:
metrics = ["Rouge-L IU Recall", "BERT IU Recall", "SBERT"]
(similarity_df
                  .groupby(["Run", "Model"])[metrics]
                  .describe()
                  .reset_index()
                #   .assign(Similarity_Score = lambda x: (x["Rouge-L IU Recall"] * 0.5 + x["BERT IU Recall"] * 0.5 + x["SBERT"]) / 2)
                  .sort_values([("Model",)])
).to_clipboard(index = False)

In [23]:
metrics = ["Rouge-L IU Recall", "BERT IU Recall", "SBERT"]

highest_scores = (similarity_df
                  .groupby(["Run", "Model"])[metrics]
                  .mean()
                  .reset_index()
                  .assign(Similarity_Score = lambda x: (x["Rouge-L IU Recall"] * 0.5 + x["BERT IU Recall"] * 0.5 + x["SBERT"]) / 2)
                  .sort_values(["Model", "Similarity_Score"], ascending = [True, False])
                  .reset_index(drop = True).drop_duplicates("Model")
                  )
highest_scores = highest_scores[~highest_scores["Model"].isin(["Claude Opus 4.6", "Claude Sonnet 4.5 Extended"])]
highest_scores

,Run,Model,Rouge-L IU Recall,BERT IU Recall,SBERT,Similarity_Score
0,Run 2,ChatGPT 5.2 Extended Thinking,0.457583,0.861500,0.612111,0.635826
4,Run 3,Claude Opus 4.6 Reasoning,0.529611,0.857222,0.591833,0.642625
10,Run 1,Gemini 3 Pro,0.406722,0.865111,0.585806,0.610861


In [24]:
highest_similarity_df = similarity_df.merge(highest_scores[["Run", "Model"]], how = "inner")
highest_similarity_df.loc[highest_similarity_df["Category"] == "How (global model-wide explanation)", "Category"] = "How (global)"
highest_similarity_df

,Run,Model,Category,Question,Rouge-L IU Recall,BERT IU Recall,BERT Precision,BERT Recall,BERT F1,SBERT
0,Run 1,Gemini 3 Pro,Data,What kind of data was the system trained on?,1.000,0.856,0.893,0.866,0.880,0.636
1,Run 1,Gemini 3 Pro,Data,What is the source of the training data?,0.730,0.899,0.870,0.892,0.881,0.839
2,Run 1,Gemini 3 Pro,Data,How were the labels/ground-truth produced?,0.750,0.855,0.918,0.912,0.915,0.779
3,Run 1,Gemini 3 Pro,Data,What is the sample size of the training data?,0.500,0.816,0.919,0.931,0.925,0.866
4,Run 1,Gemini 3 Pro,Data,What dataset(s) is the system NOT using?,0.067,0.854,0.819,0.844,0.832,0.541
...,...,...,...,...,...,...,...,...,...,...
103,Run 3,Claude Opus 4.6 Reasoning,Others,Why is the system using or not using a given a...,0.405,0.875,0.883,0.881,0.882,0.838
104,Run 3,Claude Opus 4.6 Reasoning,Others,What are the results of other people using the...,0.500,0.827,0.850,0.874,0.862,0.407
105,Run 3,Claude Opus 4.6 Reasoning,Others,Who is responsible for this system? Who are th...,1.000,0.908,0.866,0.934,0.899,0.587
106,Run 3,Claude Opus 4.6 Reasoning,UI,What are the affordances of the system?,0.700,0.848,0.834,0.876,0.855,0.542


In [25]:
def pull_accuracy_scores(response, model = None):
    results = []
    n = 180
    for category in response["questions"].keys():
        for question in response["questions"][category].keys():
            answer = response["questions"][category][question]["answer"]
            clear_output()
            print(f"{question}\n{"=" * n}\n{'\n'.join(textwrap.wrap(answer, n))}")
            x = input("Correct?")
            results += [pd.DataFrame({
                "Model": model,
                "Category": category,
                "Question": question,
                "Accuracy": [float(x)]
            })]
    clear_output()
    return pd.concat(results)

In [26]:
# chatgpt_accuracy = pull_accuracy_scores(llm_responses["run_2"]["xaiqb_chatgpt_5.2_extended_thinking"], model = "ChatGPT 5.2 Extended Thinking")

In [27]:
# chatgpt_accuracy.groupby("Category")["Accuracy"].sum()

In [28]:
# chatgpt_accuracy["Accuracy"].sum()

In [29]:
# claude_accuracy = pull_accuracy_scores(llm_responses["run_3"]["xaiqb_claude_opus_4.6_reasoning"], model = "Claude Opus 4.6 Reasoning")

In [30]:
# claude_accuracy.groupby("Category")["Accuracy"].sum()

In [31]:
# claude_accuracy["Accuracy"].sum()

In [32]:
# gemini_accuracy = pull_accuracy_scores(llm_responses["run_1"]["xaiqb_gemini_3_pro"], model = "Gemini 3 Pro")

In [33]:
# gemini_accuracy.groupby("Category")["Accuracy"].sum()

In [34]:
# gemini_accuracy["Accuracy"].sum()

In [35]:
# accuracy_scores = pd.concat([
#     chatgpt_accuracy,
#     claude_accuracy,
#     gemini_accuracy
# ])

# accuracy_scores.loc[accuracy_scores["Category"] == "How (global model-wide explanation)", "Category"] = "How (global)"
# accuracy_scores.head()

In [39]:
accuracy_scores = pd.read_excel("/Users/henry.el-jawhari/GitHub/msc_thesis/h1n1/Similarity Readability Score Results.xlsx", 
                                usecols = ["Run", "Model", "Category", "Question", "Accuracy"])
accuracy_scores.head()

,Run,Model,Category,Question,Accuracy
0,Run 2,ChatGPT 5.2 Extended Thinking,How (global),What kind of algorithm is used?,1.0
1,Run 2,ChatGPT 5.2 Extended Thinking,Data,What is the source of the training data?,1.0
2,Run 2,ChatGPT 5.2 Extended Thinking,How (global),What are the top rules/features that determine...,1.0
3,Run 2,ChatGPT 5.2 Extended Thinking,Output,What is the source of the information object(s)?,1.0
4,Run 2,ChatGPT 5.2 Extended Thinking,Others,Who is responsible for this system? Who are th...,1.0


In [40]:
highest_similarity_df = (
    highest_similarity_df
    .merge(accuracy_scores)
    .assign(**{"Similarity Score": lambda x: (x["Accuracy"] + (x["Rouge-L IU Recall"] + x["BERT IU Recall"]) / 2 + x["SBERT"]) / 3})
    )

highest_similarity_df.head()

,Run,Model,Category,Question,Rouge-L IU Recall,BERT IU Recall,BERT Precision,BERT Recall,BERT F1,SBERT,Accuracy,Similarity Score
0,Run 1,Gemini 3 Pro,Data,What kind of data was the system trained on?,1.000,0.856,0.893,0.866,0.880,0.636,1.0,0.854667
1,Run 1,Gemini 3 Pro,Data,What is the source of the training data?,0.730,0.899,0.870,0.892,0.881,0.839,1.0,0.884500
2,Run 1,Gemini 3 Pro,Data,How were the labels/ground-truth produced?,0.750,0.855,0.918,0.912,0.915,0.779,1.0,0.860500
3,Run 1,Gemini 3 Pro,Data,What is the sample size of the training data?,0.500,0.816,0.919,0.931,0.925,0.866,1.0,0.841333
4,Run 1,Gemini 3 Pro,Data,What dataset(s) is the system NOT using?,0.067,0.854,0.819,0.844,0.832,0.541,0.5,0.500500


In [ ]:
# cross_encoder_scores_df = []
# for category in xaiqb_human["questions"].keys():
#     for question in xaiqb_human["questions"][category].keys():
#         if question == "Who is responsible for this system? Who are the authors?":
#                 continue
        
#         cross_encoder_scores_df += [ pd.concat(
#     [
#         pd.DataFrame(
#     cross_encoder_model.rank(
#         query = xaiqb_human["questions"][category][question]["answer"],
#         documents = [
#             xaiqb_claude["questions"][category][question]["answer"],
#             xaiqb_chatgpt["questions"][category][question]["answer"],
#             xaiqb_gemini["questions"][category][question]["answer"]
#             ]
#         )
#     ).set_index("corpus_id").T.reset_index(drop = True)[[0, 1, 2]].rename(columns = {0: "Claude", 1: "ChatGPT", 2: "Gemini"}),
#     pd.DataFrame({"question": [question], "category": [category]})
#     ], axis = 1
# )
#                ]
        
# cross_encoder_scores_df = pd.concat(cross_encoder_scores_df)
# cross_encoder_scores_df.to_clipboard()

#### Similarity Plots

##### Similarity

In [41]:
colour_mapping = {
    "Gemini 3 Pro": "",
    "ChatGPT 5.2 Extended Thinking": "",
    "Claude Opus 4.6 Reasoning": "",
}

In [42]:
id_vars = ["Category", "Model"]
value_vars = ["Rouge-L IU Recall", "BERT IU Recall", "SBERT"] # , "Similarity Score"

In [72]:
fig = px.box(highest_similarity_df.melt(id_vars = id_vars, value_vars = value_vars, var_name = "Metric", value_name = "Score"),
             x = "Score", facet_col = "Metric", 
             color = "Model", points = "all",
             template = "simple_white",
             color_discrete_map = color_map,
             width = 1000,
             title = "Overall Similarity Metrics between Human and Large Language Model XAIQB Answers"
             
             ).for_each_xaxis(lambda x: x.update(showticklabels = True)).for_each_xaxis(lambda x: x.update(showticklabels = True)).update_layout(
                 title = dict(x = 0.5, y = 0.92), 
                 legend = dict(orientation = "h", yanchor = "top", y = 1.3, xanchor = "center", x = 0.5), 
                 margin = dict(t = 140, pad = 0),
                 boxgroupgap = 0.5)

for annotation in fig["layout"]["annotations"]: 
    annotation["textangle"] = 0

fig.for_each_annotation(lambda x: x.update(text = x.text.split("=")[1]))
fig.for_each_xaxis(lambda x: x.update(showticklabels = True))
fig.show()

In [ ]:
# fig = px.box(similarity_df[similarity_df["Model"].isin(["Claude Opus 4.6 Reasoning", "Gemini 3 Pro", "ChatGPT 5.2 Extended Thinking"])].melt(id_vars = id_vars + ["Run"], value_vars = value_vars + ["Total"], var_name = "Metric", value_name = "Score"),
#              x = "Score", facet_row = "Metric", facet_col = "Model", 
#              color = "Run", points = "all",
#              template = "simple_white", height = 1000,
#              title = "XAIQB Overall Similarity Metrics between Human and Large Language Model Answers"
#              ).for_each_xaxis(lambda x: x.update(showticklabels = True)).update_layout(title_x = 0.5, boxgroupgap = 0.5)

# for annotation in fig["layout"]["annotations"]: 
#     annotation["textangle"] = 0

# fig.for_each_annotation(lambda x: x.update(text = x.text.split("=")[1]))
# fig.for_each_xaxis(lambda x: x.update(showticklabels = True))
# fig.show()

In [44]:
fig = px.box(highest_similarity_df.melt(id_vars = id_vars, value_vars = value_vars, var_name = "Metric", value_name = "Score"),
             x = "Score", facet_row = "Category", facet_col = "Metric", 
             color = "Model", points = "all",
             height = 1000, template = "simple_white",
             title = "Category-level Similarity Metrics between Human and Large Language Model XAIQB Answers"
             ).for_each_xaxis(lambda x: x.update(showticklabels = True)).update_layout(
                 title = dict(x = 0.5, y = 0.97), 
                 legend = dict(orientation = "h", yanchor = "top", y = 1.1, xanchor = "center", x = 0.5), 
                 margin = dict(t = 150, pad = 0),
                 boxgroupgap = 0.5)

for annotation in fig["layout"]["annotations"]: 
    annotation["textangle"] = 0

fig.for_each_annotation(lambda x: x.update(text = x.text.split("=")[1]))
fig.for_each_xaxis(lambda x: x.update(showticklabels = True))
fig.show()

In [45]:
model_order = ["Gemini 3 Pro", "ChatGPT 5.2 Extended Thinking", "Claude Opus 4.6 Reasoning"]
category_order = ["Data", "Output", "Performance", "How (global)", "Others", "UI"]

to_plot = (highest_similarity_df
           .groupby(["Model", "Category"])
           .agg({x: "mean" for x in value_vars})
           .reset_index()
           .melt(id_vars = id_vars, var_name = "Metric", value_name = "Score")
           .set_index("Model").loc[model_order].reset_index()
           .set_index("Category").loc[category_order].reset_index()
           )

for metric in value_vars:

    width = 500#700 if metric == "Similarity Score" else 500
    showlegend = False#True if metric == "Similarity Score" else False
    title_x = 0.5#0.28 if metric == "Similarity Score" else  0.5

    px.line_polar(to_plot[to_plot["Metric"] == metric], 
                  theta = "Category", r = "Score", 
                  color = "Model", line_close = True, 
                  title = metric,
                  template = "simple_white",
                  width = width,
                  range_r = [0, 1]).update_layout(
                      title = dict(x = title_x, y = 0.95), 
                      showlegend = showlegend,
                    #   legend = dict(orientation = "h", yanchor = "top", y = 1.3, xanchor = "center", x = 0.5), 
                      margin = dict(pad = 0),
                      boxgroupgap = 0.5).show()

In [ ]:
# from plotly.subplots import make_subplots
# import plotly.graph_objects as go

# metrics = to_plot["Metric"].unique()
# n = len(metrics)
# colors = {"Gemini 3 Pro": "#4285F4", 
#           "ChatGPT 5.2 Extended Thinking": "#10A37F", 
#           "Claude Opus 4.6 Reasoning": "#D97706"}

# fig = make_subplots(
#     rows = 1, cols = n,
#     specs = [[{"type": "polar"}] * n],
#     subplot_titles = [f"{m}" for m in metrics]
# )

# for i, metric in enumerate(metrics, 1):
#     df_metric = to_plot[to_plot["Metric"] == metric]
#     for model in model_order:
#         df_model = df_metric[df_metric["Model"] == model]
#         r = df_model["Score"].tolist() + [df_model["Score"].tolist()[0]]
#         theta = category_order
#         fig.add_trace(
#             go.Scatterpolar(
#                 r = r, theta = theta, mode = "lines",
#                 name = model, line = dict(color=colors[model]),
#                 legendgroup = model, showlegend = (i == 1)
#             ),
#             row = 1, col = i
#         )
#     fig.update_polars(radialaxis = dict(range = [0, 1]), selector = dict(domain = fig.get_subplot(1, i).domain)).update_annotations(yshift = 40) 

# fig.update_layout(
#     # title = dict(text = "Similarity Scores", x = 0.5, y = 0.95),
#     width = 3000, height = 800,
#     template = "simple_white",
#     legend = dict(orientation = "h", yanchor = "top", y = 1.2, xanchor = "center", x = 0.5),
#     margin = dict(t = 140)
# )

# fig.update_polars(
#     radialaxis=dict(range=[0, 1]),
#     angularaxis=dict(categoryorder="array", categoryarray=category_order)
# )

# fig.show()


In [47]:
highest_similarity_df.sort_values(["SBERT"], ascending = False)

,Run,Model,Category,Question,Rouge-L IU Recall,BERT IU Recall,BERT Precision,BERT Recall,BERT F1,SBERT,Accuracy,Similarity Score
73,Run 3,Claude Opus 4.6 Reasoning,Data,What is the source of the training data?,0.952,0.909,0.887,0.936,0.911,0.902,1.0,0.944167
64,Run 2,ChatGPT 5.2 Extended Thinking,How (global),How were the parameters set?,0.167,0.862,0.883,0.902,0.893,0.887,1.0,0.800500
37,Run 2,ChatGPT 5.2 Extended Thinking,Data,What is the source of the training data?,0.730,0.900,0.874,0.903,0.888,0.879,1.0,0.898000
3,Run 1,Gemini 3 Pro,Data,What is the sample size of the training data?,0.500,0.816,0.919,0.931,0.925,0.866,1.0,0.841333
1,Run 1,Gemini 3 Pro,Data,What is the source of the training data?,0.730,0.899,0.870,0.892,0.881,0.839,1.0,0.884500
...,...,...,...,...,...,...,...,...,...,...,...,...
19,Run 1,Gemini 3 Pro,Performance,What are the limitations of the system?,0.000,0.861,0.845,0.836,0.840,0.270,0.0,0.233500
87,Run 3,Claude Opus 4.6 Reasoning,Output,What logic is used for the initial output/reco...,0.111,0.867,0.810,0.829,0.819,0.237,0.5,0.408667
61,Run 2,ChatGPT 5.2 Extended Thinking,How (global),What kind of rules does it follow?,0.000,0.843,0.826,0.808,0.817,0.227,0.5,0.382833
96,Run 3,Claude Opus 4.6 Reasoning,How (global),How does it weigh different features?,0.000,0.834,0.816,0.823,0.820,0.207,0.5,0.374667


In [48]:
def readabililty_scores(text, category = None, question = None, model_name = None):
    readabililty_scores_df = pd.DataFrame({
        "Model": model_name,
        "Category": category,
        "Question": question,
        
        "Flesch Reading Ease": [textstat.flesch_reading_ease(text)],
        "SMOG Index": [textstat.smog_index(text)],
        "Coleman Liau Index": [textstat.coleman_liau_index(text)],
        "Dale Chall Readability Score": [textstat.dale_chall_readability_score(text)],
        
        # Ignore/irrelevant
        "Text Standard": textstat.text_standard(text),
        "Difficult Words": round(textstat.difficult_words(text), 3),
        "Flesch Kincaid Grade": round(textstat.flesch_kincaid_grade(text), 3),
        "Automated Readability Index": round(textstat.automated_readability_index(text), 3),
        "Linsear Write Formula": round(textstat.linsear_write_formula(text), 3),
        "Gunning Fog": round(textstat.gunning_fog(text), 3),
        "Spache Readability": round(textstat.spache_readability(text), 3)
        })

    return readabililty_scores_df.round(3)

In [49]:
def pull_readability_scores(model_name, response):
    readabililty_scores_df = []
    for category in response["questions"].keys():
        for question in response["questions"][category].keys():
            readabililty_scores_df += [
                readabililty_scores(
                    text = response["questions"][category][question]["answer"], 
                    category = category, 
                    question = question, 
                    model_name = model_name)]
    
    return pd.concat(readabililty_scores_df)

In [ ]:
# Compute for all models

In [50]:
readability_scores_df = pd.concat([
    pd.concat([pull_readability_scores(x, y) for x, y in llm_responses["run_1"].items()]),
    pd.concat([pull_readability_scores(x, y) for x, y in llm_responses["run_2"].items()]),
    pd.concat([pull_readability_scores(x, y) for x, y in llm_responses["run_3"].items()])
    ], keys = ["Run 1", "Run 2", "Run 3"]).reset_index(level = 0).rename(columns = {"level_0": "Run"}).assign(Model = lambda x: x["Model"].apply(lambda x: prettyName(x)))

In [51]:
r_value_vars = [
    # "Flesch Reading Ease", 
    # "Text Standard", "Difficult Words", 
    "SMOG Index", "Coleman Liau Index", "Dale Chall Readability Score", "Flesch Kincaid Grade", 
    #"Automated Readability Index", "Linsear Write Formula", "Gunning Fog"
    ]

llm_readability_scores_df = pd.concat([
    readability_scores_df.merge(highest_scores[["Run", "Model"]], how = "inner"),
    ]).melt(id_vars = ["Model", "Category", "Question"], value_vars = r_value_vars, var_name = "Metric", value_name = "Score")

llm_readability_scores_df.head()

,Model,Category,Question,Metric,Score
0,Gemini 3 Pro,Data,What kind of data was the system trained on?,SMOG Index,11.208
1,Gemini 3 Pro,Data,What is the source of the training data?,SMOG Index,18.244
2,Gemini 3 Pro,Data,How were the labels/ground-truth produced?,SMOG Index,13.024
3,Gemini 3 Pro,Data,What is the sample size of the training data?,SMOG Index,11.208
4,Gemini 3 Pro,Data,What dataset(s) is the system NOT using?,SMOG Index,15.248


In [ ]:
# to_group = [
#     "Model", 
#     # "Category"
#     ]
# (llm_readability_scores_df[to_group + r_value_vars]
#  .groupby(to_group)
#  .mean()
#  .reset_index()#.pivot(index = "Category", columns = "Model")
#  .round(3)
#  ).T

In [52]:
human_readability_scores_df = pull_readability_scores("Human", xaiqb_human).melt(id_vars = ["Model", "Category", "Question"], value_vars = r_value_vars, var_name = "Metric", value_name = "Score")
human_readability_scores_df.head()

,Model,Category,Question,Metric,Score
0,Human,Data,What kind of data was the system trained on?,SMOG Index,13.024
1,Human,Data,What is the source of the training data?,SMOG Index,15.903
2,Human,Data,How were the labels/ground-truth produced?,SMOG Index,13.024
3,Human,Data,What is the sample size of the training data?,SMOG Index,6.427
4,Human,Data,What dataset(s) is the system NOT using?,SMOG Index,13.024


In [ ]:
# to_group = [
#     "Model", 
#     # "Category"
#     ]
# (human_readability_scores_df[to_group + r_value_vars]
#  .groupby(to_group)
#  .mean()
#  .reset_index()#.pivot(index = "Category", columns = "Model")
#  .round(3)
#  ).T[0]

In [53]:
readability_scores_df = (llm_readability_scores_df
 .merge(human_readability_scores_df.drop("Model", axis = 1), 
        on = ["Category", "Question", "Metric"], how = "left", suffixes = [" LLM", " Human"])
 .assign(Difference = lambda x: x["Score LLM"] - x["Score Human"])
 )

In [54]:
select_metrics = ["SMOG Index", "Coleman Liau Index", "Dale Chall Readability Score", "Flesch Kincaid Grade"]

In [55]:
to_plot = pd.concat([
    llm_readability_scores_df[llm_readability_scores_df["Metric"].isin(select_metrics)].groupby(["Model", "Metric"])["Score"].mean().reset_index(),
    human_readability_scores_df[human_readability_scores_df["Metric"].isin(select_metrics)].groupby(["Model", "Metric"])["Score"].mean().reset_index()
    ])
to_plot.head()

,Model,Metric,Score
0,ChatGPT 5.2 Extended Thinking,Coleman Liau Index,16.103056
1,ChatGPT 5.2 Extended Thinking,Dale Chall Readability Score,13.054222
2,ChatGPT 5.2 Extended Thinking,Flesch Kincaid Grade,14.736583
3,ChatGPT 5.2 Extended Thinking,SMOG Index,15.300639
4,Claude Opus 4.6 Reasoning,Coleman Liau Index,17.590611


In [56]:
model_order = ["Gemini 3 Pro", "ChatGPT 5.2 Extended Thinking", "Claude Opus 4.6 Reasoning", "Human"]
to_plot = to_plot.set_index("Model").loc[model_order].reset_index()

In [63]:
color_map = {
    'Gemini 3 Pro': '#1F77B4',
    'ChatGPT 5.2 Extended Thinking': '#2CA02C',
    'Claude Opus 4.6 Reasoning': '#FF7F0E'}

px.line_polar(
    to_plot,
    r = "Score", theta = "Metric", color = "Model", line_close = True,
    width = 900,
    template = "simple_white",
    color_discrete_map = color_map
).update_layout(legend = dict(orientation = "h", yanchor = "top", y = 1.3, xanchor = "center", x = 0.5))

In [ ]:
# def to_rgba(hex_or_rgb, alpha):
#     r, g, b = mcolors.to_rgb(hex_or_rgb)
#     return f"rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {alpha})"

# metrics = select_metrics #readability_scores_df["Metric"].unique()
# models = readability_scores_df["Model"].unique()
# colors = px.colors.qualitative.D3
# color_map = {model: colors[i % len(colors)] for i, model in enumerate(models)}

# fig = go.Figure()

# for i, metric in enumerate(metrics):
#     for model in models:
#         mask = (readability_scores_df["Metric"] == metric) & (readability_scores_df["Model"] == model)
#         data = readability_scores_df.loc[mask, "Difference"]

#         fig.add_trace(go.Violin(
#             x = data,
#             name = model,
#             legendgroup = model,
#             showlegend = (i == 0),
#             side = "positive",
#             y0 = metric,
#             orientation = "h",
#             spanmode = "soft",
#             points = False,
#             scalemode = "width",
#             width = 1.5,
#             line = dict(color = color_map[model], width = 3),
#             fillcolor = to_rgba(color_map[model], 0.25),
#         ))

# fig.add_vline(x =0, line_width = 2, line_color = "black")
# fig.add_vline(x = -10, line_width = 1, line_dash = "dash", line_color = "black")
# fig.add_vline(x = 10, line_width = 1, line_dash = "dash", line_color = "black")

# fig.update_layout(
#     title = dict(text = "LLM Readability Difference to Human Answers", x = 0.5, y = 0.97), 
#     legend = dict(orientation = "h", yanchor = "top", y = 1.1, xanchor = "center", x = 0.5), 
#     margin = dict(t = 200, pad = 0),
#     height=1500,
#     xaxis_title="Difference",
#     yaxis=dict(
#         categoryorder="array",
#         categoryarray=list(metrics),
#     ),
#     violinmode="overlay",
#     legend_title="Model",
#     template="simple_white",
# ).update_xaxes(range = [-20, 30])

# fig.show()


In [ ]:
# def to_rgba(hex_or_rgb, alpha):
#     r, g, b = mcolors.to_rgb(hex_or_rgb)
#     return f"rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {alpha})"

# metrics = readability_scores_df["Metric"].unique()
# models = readability_scores_df["Model"].unique()
# colors = px.colors.qualitative.D3
# color_map = {model: colors[i % len(colors)] for i, model in enumerate(models)}

# fig = go.Figure()

# n_models = len(models)
# row_spacing = 1.25  # increase this for more space between rows
# rug_offset_step = 0.05

# for i, metric in enumerate(metrics):
#     y_pos = i * row_spacing
#     for j, model in enumerate(models):
#         mask = (readability_scores_df["Metric"] == metric) & (readability_scores_df["Model"] == model)
#         data = readability_scores_df.loc[mask, "Difference"]

#         fig.add_trace(go.Violin(
#             x=data,
#             name=model,
#             legendgroup=model,
#             showlegend=(i == 0),
#             side="positive",
#             y0=y_pos,
#             orientation="h",
#             spanmode="soft",
#             points=False,
#             scalemode="width",
#             width=1.5,
#             line=dict(color=color_map[model], width=3),
#             fillcolor=to_rgba(color_map[model], 0.25),
#         ))

#         rug_y = [y_pos - 0.05 - j * rug_offset_step] * len(data)

#         fig.add_trace(go.Scatter(
#             x=data,
#             y=rug_y,
#             mode="markers",
#             marker=dict(
#                 symbol="line-ns-open",
#                 color=color_map[model],
#                 size=8,
#                 line=dict(width=1.5),
#             ),
#             name=model,
#             legendgroup=model,
#             showlegend=False,
#         ))

# fig.add_vline(x = 0, line_width = 2, line_color = "black")
# fig.add_vline(x = -10, line_width = 1, line_dash = "dash", line_color = "black")
# fig.add_vline(x = 10, line_width = 1, line_dash = "dash", line_color = "black")

# fig.update_layout(
#     title = dict(text = "LLM Readability Difference to Human Answers", x = 0.5, y = 0.97), 
#     legend = dict(orientation = "h", yanchor = "top", y = 1.1, xanchor = "center", x = 0.5), 
#     margin = dict(t = 200, pad = 0),
#     height=1500,
#     xaxis_title="Readability Difference",
#     yaxis=dict(
#         tickvals=[i * row_spacing for i in range(len(metrics))],
#         ticktext=list(metrics),
#     ),
#     violinmode="overlay",
#     legend_title="Model",
#     template="simple_white",
# ).update_xaxes(range = [-20, 30])

# fig.show()


In [64]:
metrics

array(['SMOG Index', 'Coleman Liau Index', 'Dale Chall Readability Score',
       'Flesch Kincaid Grade'], dtype=object)

In [69]:
def to_rgba(hex_or_rgb, alpha):
    r, g, b = mcolors.to_rgb(hex_or_rgb)
    return f"rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {alpha})"

# metrics = readability_scores_df["Metric"].unique()
metrics = ['Dale Chall Readability Score', 'Flesch Kincaid Grade']
models = readability_scores_df["Model"].unique()
colors = px.colors.qualitative.D3
color_map = {model: colors[i % len(colors)] for i, model in enumerate(models)}
color_map = {
    'Gemini 3 Pro': '#1F77B4',
    'ChatGPT 5.2 Extended Thinking': '#2CA02C',
    'Claude Opus 4.6 Reasoning': '#FF7F0E'}

fig = go.Figure()

n_models = len(models)
row_spacing = 1.25
rug_offset_step = 0.05

for i, metric in enumerate(metrics):
    y_pos = i * row_spacing
    for j, model in enumerate(models):
        mask = (readability_scores_df["Metric"] == metric) & (readability_scores_df["Model"] == model)
        data = readability_scores_df.loc[mask, "Difference"]

        fig.add_trace(go.Violin(
            x=data,
            name=model,
            legendgroup=model,
            showlegend=(i == 0),
            side="positive",
            y0=y_pos,
            orientation="h",
            spanmode="soft",
            points=False,
            scalemode="width",
            width=1.5,
            line=dict(color=color_map[model], width=3),
            fillcolor=to_rgba(color_map[model], 0.25),
        ))

        # Median line
        median_val = np.mean(data) # Use median?
        fig.add_trace(go.Scatter(
            x=[median_val, median_val],
            y=[y_pos, y_pos + 0.075],
            mode="lines",
            line=dict(color=color_map[model], width=3),
            name=model,
            legendgroup=model,
            showlegend=False,
        ))

        rug_y = [y_pos - 0.1 - j * rug_offset_step] * len(data)

        fig.add_trace(go.Scatter(
            x=data,
            y=rug_y,
            mode="markers",
            marker=dict(
                symbol="line-ns-open",
                color=color_map[model],
                size=8,
                line=dict(width=1.5),
            ),
            name=model,
            legendgroup=model,
            showlegend=False,
        ))

fig.add_vline(x=0, line_width=2, line_color="black")
fig.add_vline(x=-10, line_width=1, line_dash="dash", line_color="black")
fig.add_vline(x=10, line_width=1, line_dash="dash", line_color="black")

fig.update_layout(
    title=dict(text="LLM Readability Difference to Human Answers", x=0.5, y=0.97),
    legend=dict(orientation="h", yanchor="top", y=1.1, xanchor="center", x=0.5),
    margin=dict(t=200, pad=0),
    height=1000,
    xaxis_title="Readability Difference",
    yaxis=dict(
        tickvals=[i * row_spacing for i in range(len(metrics))],
        ticktext=list(metrics),
    ),
    violinmode="overlay",
    legend_title="Model",
    template="simple_white",
).update_xaxes(range=[-20, 30])

fig.show()


In [ ]:
text_analysis = llm_responses["run_1"]["xaiqb_gemini_3_pro"]

In [ ]:
textstat.sentence_count("The toggle buttons (eg, for `doctor_recc_h1n1` or `opinion_h1n1_risk`) allow the user to change the input values for a hypothetical patient profile. The float slider adjusts the decision threshold (default 05) to see how it affects the binary prediction.")

2